# Projeto Fictus | Análise Financeira — ETL Pipeline

---

## Contexto
A análise de Vendas confirmou o crescimento da empresa-alvo e recomendou aquisição condicional. A análise de Logística definiu o modelo operacional de entrega.
Agora o Projeto Fictus avança para a terceira e última frente da análise: **examinar as atividades financeiras da empresa-alvo — estrutura de crédito, perfil de parcelamento, exposição a risco e eficiência dos intermediários financeiros — para completar a base de evidências da recomendação de aquisição.**

Este ETL carrega os dados da **análise de vendas** e deriva as variáveis financeiras necessárias para os cinco blocos analíticos das Finanças.

## Dependência
Este notebook **requer** que a análise de Vendas (00_ETL_pipeline.ipynb) tenha sido executada.
Os arquivos em `data/pre-tratados/` são o ponto de entrada obrigatório.

## Conexão com os Projetos Anteriores
| Achado nas Frentes Vendas e Logística | Pergunta que abre na Frente Finanças |
|---|---|
| Crescimento de volume com ticket médio sustentado | Quem está financiando esse crescimento — a empresa ou os intermediários? |
| Alto percentual de vendas parceladas no cartão | Qual o prazo real de recebimento e como isso pressiona o caixa? |
| Concentração geográfica SP+RJ+MG | O risco de crédito também está concentrado nessas regiões? |
| Sazonalidade operacional com picos previsíveis | Picos de venda geram picos de capital em aberto — qual a exposição? |

## Tabelas geradas
| Tabela | Descrição |
|---|---|
| `fin_fato.csv` | Fato enriquecido com variáveis financeiras: prazo de recebimento, exposição, anomalias |
| `fin_mensal.csv` | Série temporal mensal: mix de pagamento, capital em aberto, PMR, variabilidade |
| `fin_trimestral.csv` | Série trimestral para análises de tendência e break-even |
| `fin_pagamento.csv` | Agregação por modalidade de pagamento: spread estimado, prazo, concentração |
| `fin_regional.csv` | Métricas financeiras por estado do cliente: exposição, anomalias, concentração |

---


## PASSO 0 — Verificação de Dependências

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# ─── Caminhos — detecta automaticamente a pasta do notebook ──────────────────
try:
    NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path().resolve()

def _find_base(start: Path) -> Path:
    for p in [start, start.parent, start.parent.parent]:
        if (p / "data").exists() or (p / "notebooks").exists():
            return p
    return start
BASE_DIR   = _find_base(NOTEBOOK_DIR)
DIR_PRE    = BASE_DIR / "data" / "pre-tratados"
DIR_FIN    = BASE_DIR / "data" / "finance"
DIR_FIN.mkdir(parents=True, exist_ok=True)

# ─── Verifica arquivos do Projeto 1 ──────────────────────────────────────────
ARQUIVOS_NECESSARIOS = [
    "fato_vendas.csv", "dim_produto.csv", "dim_cliente.csv",
    "dim_vendedor.csv", "dim_tempo.csv",
]

print("Verificando dependências do Projeto 1...\n")
tudo_ok = True
for arq in ARQUIVOS_NECESSARIOS:
    caminho = DIR_PRE / arq
    if caminho.exists():
        kb = caminho.stat().st_size / 1024
        print(f"  ✅ {arq:<30} {kb:>8.1f} KB")
    else:
        print(f"  ❌ {arq:<30} NÃO ENCONTRADO")
        tudo_ok = False

print()
if tudo_ok:
    print("✅ Todas as dependências encontradas. Pode prosseguir.")
else:
    print("❌ Execute o Projeto 1 (01_ETL_pipeline.ipynb) antes de continuar.")
    print("   Copie a pasta data/pre-tratados/ do Projeto 1 para este projeto.")


## PASSO 1 — Carregamento dos Dados

> **Granularidade:** a tabela `fato_vendas` tem granularidade de **item de pedido** — um pedido com 3 itens gera 3 linhas. As agregações financeiras preservam essa granularidade para cálculo de spread e capital, mas usam `nunique('id_pedido')` para contagem de pedidos, evitando dupla contagem.

In [ ]:
def ler_csv(caminho, **kwargs):
    df = pd.read_csv(caminho, low_memory=False, **kwargs)
    df.columns = df.columns.str.strip()
    return df

fato  = ler_csv(DIR_PRE / "fato_vendas.csv")
dim_p = ler_csv(DIR_PRE / "dim_produto.csv")
dim_c = ler_csv(DIR_PRE / "dim_cliente.csv")
dim_v = ler_csv(DIR_PRE / "dim_vendedor.csv")
dim_t = ler_csv(DIR_PRE / "dim_tempo.csv")

# ─── Conversão de tipos ───────────────────────────────────────────────────────
for col in ["data_compra", "data_entrega_cliente", "data_previsao_entrega",
            "data_aprovacao", "data_envio_transportadora"]:
    if col in fato.columns:
        fato[col] = pd.to_datetime(fato[col], errors="coerce")
dim_t["data"] = pd.to_datetime(dim_t["data"], errors="coerce")

for col in ["preco", "valor_frete", "valor_total_item", "valor_pagamento_total",
            "lead_time_dias", "atraso_dias", "nota_review", "numero_parcelas"]:
    if col in fato.columns:
        fato[col] = pd.to_numeric(fato[col], errors="coerce")

# ─── Enriquecimento ───────────────────────────────────────────────────────────
fato = fato.merge(dim_p[["id_produto", "nome_categoria_produto"]], on="id_produto", how="left")
fato = fato.merge(dim_c[["id_cliente", "estado_cliente"]],         on="id_cliente", how="left")
fato = fato.merge(dim_v[["id_vendedor", "estado_vendedor"]],       on="id_vendedor", how="left")
fato = fato.merge(dim_t[["id_data", "ano", "mes", "trimestre", "nome_mes", "ano_mes"]],
                  on="id_data", how="left")
fato["periodo"] = fato["ano"].astype(str) + "-Q" + fato["trimestre"].astype(str)

# ─── Filtro temporal (jan/2024 em diante — mesmo padrão do Retail) ───────────
DATA_INICIO = "2024-01-01"
DATA_FIM    = "2025-08-31"   # alinhado com ETL de Vendas e Logística
fato = fato[(fato["data_compra"] >= DATA_INICIO) & (fato["data_compra"] <= DATA_FIM)].copy()
fe   = fato[fato["status_pedido"] == "entregue"].copy()

print(f"[FILTRO] Período: {DATA_INICIO} → {DATA_FIM}")
print(f"fato total    : {len(fato):>8} linhas")
print(f"fato entregues: {len(fe):>8} linhas ({len(fe)/len(fato)*100:.1f}%)")
print(f"\nModalidades de pagamento disponíveis:")
if "tipo_pagamento" in fato.columns:
    print(fato["tipo_pagamento"].value_counts().to_string())


### Reconciliação — Totais antes e após o filtro temporal

Validação de integridade: garante que o filtro temporal não introduziu distorção nos totais financeiros.

In [ ]:
# ─── Reconciliação de integridade ────────────────────────────────────────────
# Captura totais da base completa antes do filtro (recarrega só o fato bruto)
_fato_completo = ler_csv(DIR_PRE / "fato_vendas.csv")
for _col in ["preco", "valor_frete", "numero_parcelas"]:
    if _col in _fato_completo.columns:
        _fato_completo[_col] = pd.to_numeric(_fato_completo[_col], errors="coerce")

_receita_total_raw  = _fato_completo["preco"].sum()
_pedidos_total_raw  = _fato_completo["id_pedido"].nunique()
_receita_filtrada   = fato["preco"].sum()
_pedidos_filtrados  = fato["id_pedido"].nunique()
_pct_receita_ret    = _receita_filtrada / _receita_total_raw * 100
_pct_pedidos_ret    = _pedidos_filtrados / _pedidos_total_raw * 100

print("=" * 55)
print("RECONCILIAÇÃO — INTEGRIDADE DO FILTRO TEMPORAL")
print("=" * 55)
print(f"  Receita total (base completa) : R$ {_receita_total_raw:>14,.0f}")
print(f"  Receita após filtro jan/2024+ : R$ {_receita_filtrada:>14,.0f}  ({_pct_receita_ret:.1f}% retido)")
print(f"  Pedidos total (base completa) : {_pedidos_total_raw:>14,}")
print(f"  Pedidos após filtro jan/2024+ : {_pedidos_filtrados:>14,}  ({_pct_pedidos_ret:.1f}% retido)")
print(f"  Receita descartada (2023)     : R$ {_receita_total_raw - _receita_filtrada:>14,.0f}  ({100-_pct_receita_ret:.1f}% do total)")
print("")
# Validação: fato não deve ter pedidos duplicados por período
_dup_check = fato.groupby(["id_pedido", "ano_mes"]).size().max()
print(f"  Máx. registros por pedido×mês : {_dup_check} ",
      "(esperado: >1 para pedidos com múltiplos itens — ok)" if _dup_check > 1 else "")
print("=" * 55)
del _fato_completo  # liberar memória


## PASSO 2 — Derivação de Variáveis Financeiras

> *As variáveis financeiras são derivadas a partir das informações transacionais disponíveis no dataset. O dataset Olist registra o frete pago pelo cliente e o tipo/parcelamento do pagamento — mas não o custo de repasse à operadora nem a inadimplência real. As variáveis de risco são construídas como proxies analíticos com premissas declaradas e auditáveis.*
>
> **Limitação do `capital_em_aberto`:** a variável é calculada como snapshot no momento da compra — assume que todas as parcelas futuras estão abertas simultaneamente. No acumulado do período, isso superestima a exposição real, pois parcelas já liquidadas continuam computadas. Interpretar como exposição máxima potencial, não como saldo devedor real.

In [ ]:
# Premissas financeiras 2025 - Todas auditáveis e modificáveis
# Taxas de desconto estimadas por modalidade (média de mercado brasileiro 2025)
# Fonte: Banco Central do Brasil - Estatísticas de Pagamentos de Varejo e de Cartões (Dados 2025)
# Link Ref: https://www.bcb.gov.br/estatisticas/spbadendos (Portal de Dados Abertos)

TAXA_DESCONTO = {
    "cartao_credito": 0.0226,  # 2,26% MDR médio (Fonte: BCB/Dados Abertos 2025) + custo de parcelamento
    "boleto":         0.0085,  # 0,85% ao mês (Tarifa média ponderada para emissão e liquidação)
    "voucher":        0.0045,  # 0,45% ao mês (Menor risco e spreads competitivos para benefícios)
    "cartao_debito":  0.0108,  # 1,08% ao mês (MDR médio para função débito - Fonte: BCB 2025)
    "nao_definido":   0.0150,  # 1,50% ao mês (Estimativa conservadora para manutenção de margem)
}

# Prazo médio de recebimento por modalidade (dias corridos) em 2025
PMR_BASE = {
    "cartao_credito": 28.0,    # Redução gradual do ciclo D+30 para D+28 no mercado
    "boleto":          2.0,    # Compensação média D+1 a D+2 (processamento interbancário moderno)
    "voucher":         1.0,    # Liquidação rápida em cartões de benefício
    "cartao_debito":   1.0,    # D+1 padrão de mercado em 2025
    "nao_definido":   15.0,    # Estimativa central
}


# Threshold de anomalia de pagamento (proxy de inadimplência)
# Pedidos cancelados, não entregues ou com pagamento nao_definido são sinais de risco latente
STATUS_ANOMALIA = ["cancelado", "unavailable", "nao_definido"]

print("Premissas financeiras carregadas:")
print(f"  Modalidades mapeadas: {list(TAXA_DESCONTO.keys())}")
print(f"  Taxa máxima estimada: {max(TAXA_DESCONTO.values())*100:.2f}% a.m.")
print(f"  Taxa mínima estimada: {min(TAXA_DESCONTO.values())*100:.2f}% a.m.")


In [ ]:
# ─── Variáveis financeiras por transação ─────────────────────────────────────

# Prazo médio de recebimento ajustado pelo número de parcelas
# Parcelas adicionais além da primeira estendem o PMR proporcionalmente
fato["pmr_dias"] = fato["tipo_pagamento"].map(PMR_BASE).fillna(15.0)
fato["pmr_ajustado"] = np.where(
    fato["tipo_pagamento"] == "cartao_credito",
    fato["pmr_dias"] * fato["numero_parcelas"].fillna(1).clip(lower=1),
    fato["pmr_dias"]
)
fato["pmr_ajustado"] = fato["pmr_ajustado"].clip(upper=360)  # cap em 12 meses

# Taxa de desconto por modalidade
fato["taxa_desconto"] = fato["tipo_pagamento"].map(TAXA_DESCONTO).fillna(0.015)

# Spread estimado capturado por intermediários (em R$)
# = valor da venda × taxa mensal × (PMR ajustado / 30)
fato["spread_intermediario"] = (
    fato["preco"] * fato["taxa_desconto"] * (fato["pmr_ajustado"] / 30)
)

# Capital em aberto estimado (valor vendido ainda não recebido)
# = valor da venda (proxy: considera que parcelas futuras ainda estão abertas)
fato["capital_em_aberto"] = np.where(
    fato["tipo_pagamento"] == "cartao_credito",
    fato["preco"] * ((fato["numero_parcelas"].fillna(1) - 1) / fato["numero_parcelas"].fillna(1).clip(lower=1)),
    0.0
)

# Flag de anomalia de pagamento (proxy de risco de crédito)
fato["anomalia_pagamento"] = (
    fato["status_pedido"].isin(STATUS_ANOMALIA) |
    (fato["tipo_pagamento"] == "nao_definido")
).astype(int)

# Faixa de parcelamento
def classificar_parcelas(n):
    if pd.isna(n) or n <= 1:  return "a_vista"
    elif n <= 3:               return "2x_3x"
    elif n <= 6:               return "4x_6x"
    elif n <= 12:              return "7x_12x"
    else:                      return "13x_mais"

fato["faixa_parcelamento"] = fato["numero_parcelas"].apply(classificar_parcelas)

print("Variáveis financeiras criadas:")
print(f"  pmr_ajustado         : prazo médio de recebimento ajustado por parcelas (dias)")
print(f"  taxa_desconto        : taxa estimada capturada por intermediários (% a.m.)")
print(f"  spread_intermediario : spread estimado em R$ por transação")
print(f"  capital_em_aberto    : capital comprometido em parcelas futuras (R$)")
print(f"  anomalia_pagamento   : flag de risco de crédito latente (0/1)")
print(f"  faixa_parcelamento   : classificação por número de parcelas")
print(f"\nSpread total estimado : R$ {fato['spread_intermediario'].sum():,.0f}")
print(f"Capital em aberto     : R$ {fato['capital_em_aberto'].sum():,.0f}")
print(f"Anomalias identificadas: {fato['anomalia_pagamento'].sum():,} pedidos")


## PASSO 3 — Geração das Tabelas Analíticas

In [ ]:
# ─── fin_mensal: série temporal para análise de PMR, capital e variabilidade ─
fin_mensal = (
    fato.groupby("ano_mes")
    .agg(
        n_pedidos            = ("id_pedido",            "nunique"),
        receita_total        = ("preco",                "sum"),
        frete_total          = ("valor_frete",          "sum"),
        spread_total         = ("spread_intermediario", "sum"),
        capital_aberto       = ("capital_em_aberto",    "sum"),
        pmr_medio            = ("pmr_ajustado",         "mean"),
        n_anomalias          = ("anomalia_pagamento",   "sum"),
        ticket_medio         = ("preco",                "mean"),
        parcelas_medio       = ("numero_parcelas",      "mean"),
    )
    .reset_index().sort_values("ano_mes")
)
fin_mensal["pct_anomalia"]         = fin_mensal["n_anomalias"] / fin_mensal["n_pedidos"] * 100
fin_mensal["pct_spread_receita"]   = fin_mensal["spread_total"] / fin_mensal["receita_total"] * 100
fin_mensal["pct_capital_receita"]  = fin_mensal["capital_aberto"] / fin_mensal["receita_total"] * 100
fin_mensal["razao_financiamento"]  = fin_mensal["capital_aberto"] / fin_mensal["receita_total"]
fin_mensal["cv_receita"]           = fin_mensal["receita_total"].expanding().std() / fin_mensal["receita_total"].expanding().mean()

# ─── fin_trimestral: agregação trimestral ────────────────────────────────────
fin_trim = (
    fato.groupby("periodo")
    .agg(
        n_pedidos        = ("id_pedido",            "nunique"),
        receita_total    = ("preco",                "sum"),
        spread_total     = ("spread_intermediario", "sum"),
        capital_aberto   = ("capital_em_aberto",    "sum"),
        pmr_medio        = ("pmr_ajustado",         "mean"),
        n_anomalias      = ("anomalia_pagamento",   "sum"),
        ticket_medio     = ("preco",                "mean"),
    )
    .reset_index().sort_values("periodo")
)
fin_trim["pct_anomalia"]       = fin_trim["n_anomalias"] / fin_trim["n_pedidos"] * 100
fin_trim["pct_spread_receita"] = fin_trim["spread_total"] / fin_trim["receita_total"] * 100

# ─── fin_pagamento: métricas por modalidade de pagamento ─────────────────────
fin_pag = (
    fato.groupby("tipo_pagamento")
    .agg(
        n_pedidos         = ("id_pedido",            "nunique"),
        receita_total     = ("preco",                "sum"),
        spread_total      = ("spread_intermediario", "sum"),
        capital_aberto    = ("capital_em_aberto",    "sum"),
        pmr_medio         = ("pmr_ajustado",         "mean"),
        parcelas_medio    = ("numero_parcelas",      "mean"),
        n_anomalias       = ("anomalia_pagamento",   "sum"),
        ticket_medio      = ("preco",                "mean"),
    )
    .reset_index().sort_values("receita_total", ascending=False)
)
fin_pag["pct_receita"]     = fin_pag["receita_total"] / fin_pag["receita_total"].sum() * 100
fin_pag["pct_acum"]        = fin_pag["pct_receita"].cumsum()
fin_pag["pct_spread"]      = fin_pag["spread_total"] / fin_pag["receita_total"] * 100
fin_pag["pct_anomalia"]    = fin_pag["n_anomalias"] / fin_pag["n_pedidos"] * 100
fin_pag["taxa_desconto"]   = fin_pag["tipo_pagamento"].map(TAXA_DESCONTO).fillna(0.015) * 100

# ─── fin_regional: métricas por estado do cliente ────────────────────────────
fin_reg = (
    fato.groupby("estado_cliente")
    .agg(
        n_pedidos         = ("id_pedido",            "nunique"),
        receita_total     = ("preco",                "sum"),
        spread_total      = ("spread_intermediario", "sum"),
        capital_aberto    = ("capital_em_aberto",    "sum"),
        pmr_medio         = ("pmr_ajustado",         "mean"),
        n_anomalias       = ("anomalia_pagamento",   "sum"),
        ticket_medio      = ("preco",                "mean"),
        parcelas_medio    = ("numero_parcelas",      "mean"),
    )
    .reset_index().sort_values("receita_total", ascending=False)
)
fin_reg["pct_receita"]   = fin_reg["receita_total"] / fin_reg["receita_total"].sum() * 100
fin_reg["pct_acum"]      = fin_reg["pct_receita"].cumsum()
fin_reg["pct_anomalia"]  = fin_reg["n_anomalias"] / fin_reg["n_pedidos"] * 100
fin_reg["pct_spread"]    = fin_reg["spread_total"] / fin_reg["receita_total"] * 100

# ─── fin_faixa: métricas por faixa de parcelamento ───────────────────────────
fin_faixa = (
    fato.groupby("faixa_parcelamento")
    .agg(
        n_pedidos       = ("id_pedido",            "nunique"),
        receita_total   = ("preco",                "sum"),
        spread_total    = ("spread_intermediario", "sum"),
        capital_aberto  = ("capital_em_aberto",    "sum"),
        pmr_medio       = ("pmr_ajustado",         "mean"),
        ticket_medio    = ("preco",                "mean"),
        n_anomalias     = ("anomalia_pagamento",   "sum"),
    )
    .reset_index()
)
fin_faixa["pct_receita"]  = fin_faixa["receita_total"] / fin_faixa["receita_total"].sum() * 100
fin_faixa["pct_spread"]   = fin_faixa["spread_total"] / fin_faixa["receita_total"] * 100
fin_faixa["pct_anomalia"] = fin_faixa["n_anomalias"] / fin_faixa["n_pedidos"] * 100

# ─── fin_fato: fato enriquecido com variáveis financeiras ────────────────────
colunas_fin = [
    "id_pedido", "id_produto", "id_cliente", "id_vendedor", "id_data",
    "data_compra", "data_aprovacao", "data_entrega_cliente", "data_previsao_entrega",
    "preco", "valor_frete", "valor_pagamento_total",
    "tipo_pagamento", "numero_parcelas", "faixa_parcelamento",
    "pmr_ajustado", "taxa_desconto", "spread_intermediario", "capital_em_aberto",
    "anomalia_pagamento", "status_pedido", "nota_review",
    "lead_time_dias", "atraso_dias", "entregue_no_prazo",
    "nome_categoria_produto", "estado_cliente", "estado_vendedor",
    "ano", "mes", "trimestre", "nome_mes", "ano_mes", "periodo"
]
fin_fato = fato[[c for c in colunas_fin if c in fato.columns]].copy()

print("Tabelas geradas:")
print(f"  fin_fato       : {len(fin_fato):>8} linhas | {len(fin_fato.columns)} colunas")
print(f"  fin_mensal     : {len(fin_mensal):>8} linhas")
print(f"  fin_trimestral : {len(fin_trim):>8} linhas")
print(f"  fin_pagamento  : {len(fin_pag):>8} linhas | {fin_pag['tipo_pagamento'].nunique()} modalidades")
print(f"  fin_regional   : {len(fin_reg):>8} linhas | {fin_reg['estado_cliente'].nunique()} estados")
print(f"  fin_faixa      : {len(fin_faixa):>8} linhas | faixas de parcelamento")


## PASSO 4 — Exportação

In [ ]:
tabelas = {
    "fin_fato.csv"        : fin_fato,
    "fin_mensal.csv"      : fin_mensal,
    "fin_trimestral.csv"  : fin_trim,
    "fin_pagamento.csv"   : fin_pag,
    "fin_regional.csv"    : fin_reg,
    "fin_faixa.csv"       : fin_faixa,
}

for nome, df in tabelas.items():
    df.to_csv(DIR_FIN / nome, index=False, encoding="utf-8", sep=",", decimal=".")
    print(f"  ✅ {nome:<25} → {len(df):>6} linhas")

print(f"\nExportado em: {DIR_FIN}")
print("\nETL Finance concluído. Execute os notebooks de análise em sequência:")
print("  01_perfil_pagamento.ipynb")
print("  02_inadimplencia_risco.ipynb")
print("  03_rentabilidade_financeira.ipynb")
print("  04_capital_escalabilidade.ipynb")
print("  05_cenarios_recomendacao_finance.ipynb")


---

> Esta análise faz parte do **Projeto Fictus**, conduzido pela Lufi Data Consulting. Os três módulos analíticos — Vendas, Logística e Finanças — compõem a base do Relatório de Recomendação de Aquisição.